# TSA_ch12_gdp

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** US Real GDP spectral analysis — periodogram and multitaper estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.signal import periodogram
from scipy.signal.windows import dpss

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
data = sm.datasets.macrodata.load_pandas().data
gdp = np.log(data['realgdp'].values)
T = len(gdp)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(gdp, color=COLORS['blue'], lw=1.5)
ax.set_title('US Real GDP (log, quarterly)')
ax.set_xlabel('Quarter (1959Q1 onwards)')
ax.set_ylabel('log Real GDP')
fig.patch.set_alpha(0); ax.set_facecolor('none')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Multitaper periodogram of log GDP
tapers, _ = dpss(T, NW=4, Kmax=7, return_ratios=True)
mt_psds = []
for tap in tapers:
    ft = np.fft.rfft(gdp * tap)
    mt_psds.append(np.abs(ft)**2 / T)
p_mt = np.mean(mt_psds, axis=0)
f_mt = np.fft.rfftfreq(T, d=1)  # cycles/quarter

f_raw, p_raw = periodogram(gdp)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f_raw[1:], p_raw[1:], color=COLORS['gray'], lw=0.8, alpha=0.5, label='Raw periodogram')
ax.semilogy(f_mt[1:], p_mt[1:], color=COLORS['blue'], lw=2, label='Multitaper (NW=4, K=7)')
# Business cycle band 6-32 quarters
ax.axvspan(1/32, 1/6, alpha=0.15, color=COLORS['amber'], label='Business cycle (6–32q)')
ax.set_xlabel('Frequency (cycles/quarter)')
ax.set_ylabel('PSD (log)')
ax.set_title('Multitaper Spectral Estimate of US Real GDP')
fig.patch.set_alpha(0); ax.set_facecolor('none')
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig('gdp_periodogram.pdf', bbox_inches='tight')
plt.show()